# Step 06 - LLM Based Response Generation

## Objective

In this notebook we implement a Large Language Model based
Natural Language Generation system.

Input:

Intent + Slots

Example:

intent = order_status
order_id = ORD123

Output:

I can help with order status for order ORD123.


## Why LLM?

The custom Decoder Transformer trained earlier learns patterns
from limited training data.

However, it struggles with:

- unseen slot values
- copying IDs
- generalization

LLMs are pretrained on massive text corpora and can perform
instruction following using prompts.

We evaluate whether LLM improves:

- Response quality
- Slot preservation
- Hallucination reduction

In [1]:
#CELL 2 - Imports

import json
import os
import time
from tqdm import tqdm

In [2]:
#CELL 3 - Load Validation Data

DATA_PATH = "../data/"


with open(
    DATA_PATH + "linearized_validation.json"
) as f:

    val_data = json.load(f)


print(
    "Validation Samples:",
    len(val_data)
)

Validation Samples: 60


In [3]:
# CELL 4 - Understand One Sample

sample = val_data[0]

sample

{'sample_id': 'SMP0154',
 'intent': 'order_status',
 'slots': {'order_id': 'ORD841324',
  'phone_last4': '9488',
  'channel': 'website'},
 'linearized_sequence': '[BOS] intent : order_status <sep> channel : website <sep> order_id : ORD841324 <sep> phone_last4 : 9488 [EOS]',
 'target_text': 'I can help with order status for order ORD841324.',
 'edge_case': False,
 'edge_case_types': []}

In [4]:
#CELL 5 - Prompt Builder

def build_prompt(sample):


    prompt = f"""

You are a customer service response generation assistant.


Task:
Generate a natural language response using ONLY the provided intent and slots.


Rules:

1. Preserve all IDs exactly.
2. Do not invent missing information.
3. Do not add extra details.
4. Generate one short customer support response.



Input:

{sample["linearized_sequence"]}



Response:

"""


    return prompt

In [ ]:
# Testing above prompt 
# 
print(
    build_prompt(
        val_data[0]
    )
)



You are a customer service response generation assistant.


Task:
Generate a natural language response using ONLY the provided intent and slots.


Rules:

1. Preserve all IDs exactly.
2. Do not invent missing information.
3. Do not add extra details.
4. Generate one short customer support response.



Input:

[BOS] intent : order_status <sep> channel : website <sep> order_id : ORD841324 <sep> phone_last4 : 9488 [EOS]



Response:




#CELL 6 - Install LLM Library   
# For Assignment 
#For assignment, easiest is HuggingFace.

#In terminal:

#pip install transformers accelerate sentencepiece

#Then:

#pip freeze > requirements.txt

In [10]:
# import transformers

# print(transformers.__version__)

# import sys

# print(sys.executable)
# print(sys.version)

# import torch

# print(torch.__version__)

# import numpy as np
# print(np.__version__)

import sys

print(sys.executable)

/Users/pavankumararepu/BitsPilani/2nd Year/AIMLCZG521_Con_AI/Assignment1/AssignmentSolution/venv/bin/python


In [4]:
#CELL 7 - Load Small LLM

# from transformers import pipeline


# generator = pipeline(
#     task="text2text-generation",
#     model="google/flan-t5-small"
# )


# print("LLM Loaded Successfully")

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


MODEL_NAME = "google/flan-t5-small"


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)


print("LLM Loaded")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

ImportError: 
AutoModelForSeq2SeqLM requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [3]:
# CELL 8 - Generate Single Response 

prompt = build_prompt(
    val_data[0]
)


result = generator(

    prompt,

    max_length=80,

    do_sample=False

)


print(
    result[0]["generated_text"]
)

NameError: name 'build_prompt' is not defined

In [ ]:
#CELL 9 — Generate All Validation Responses

llm_predictions=[]


start=time.time()


for item in tqdm(val_data):


    prompt = build_prompt(
        item
    )


    result = generator(

        prompt,

        max_length=80,

        do_sample=False

    )


    llm_predictions.append(

        {
            "input":
            item["linearized_sequence"],


            "expected":
            item["target_text"],


            "generated":
            result[0]["generated_text"]

        }

    )


end=time.time()


print(
    "Total time:",
    end-start
)

  0%|          | 0/60 [00:00<?, ?it/s]


NameError: name 'generator' is not defined

In [ ]:
#CELL 7 - Load Small LLM

# from transformers import pipeline


# generator = pipeline(
#     task="text2text-generation",
#     model="google/flan-t5-small"
# )


# print("LLM Loaded Successfully")

In [ ]:
# CELL 10 - Save Output

os.makedirs(
    "../outputs",
    exist_ok=True
)


with open(

    "../outputs/llm_predictions.json",

    "w"

) as f:


    json.dump(

        llm_predictions,

        f,

        indent=4

    )


print(
    "Saved:",
    len(llm_predictions)
)

In [ ]:
# CELL 11 - Compare Examples

for i in range(5):


    print("="*60)


    print(
        "INPUT:"
    )

    print(
        llm_predictions[i]["input"]
    )


    print(
        "\nEXPECTED:"
    )

    print(
        llm_predictions[i]["expected"]
    )


    print(
        "\nLLM:"
    )

    print(
        llm_predictions[i]["generated"]
    )

/Users/pavankumararepu/BitsPilani/2nd Year/AIMLCZG521_Con_AI/Assignment1/AssignmentSolution/venv/bin/python
NumPy 2.4.6
zsh:1: no such file or directory: /Users/pavankumararepu/BitsPilani/2nd
